# Eco Travel Advisor for Google Colab

This notebook recreates the **Eco Travel Advisor** Rasa chatbot inside Google Colab, installs the required dependencies, trains the assistant, runs the custom action server, starts the Rasa REST API, launches the Streamlit frontend, and exposes the UI from the notebook.

Run the cells from top to bottom. All project `.py` files are created with `%%writefile` as requested.


## Notebook Steps

1. Prepare a Colab-compatible Python environment.
2. Recreate the project files under `/content/eco_travel_advisor_project`.
3. Optionally load API credentials from Colab Secrets into `.env`.
4. Install Python dependencies.
5. Validate training data, run tests, and train the model.
6. Start the action server, Rasa server, and Streamlit frontend.
7. Open the Streamlit app directly from Colab.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import urllib.request

PROJECT_ROOT = Path('/content/eco_travel_advisor_project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

for relative_dir in [
    'actions',
    'actions/mock_data',
    'data',
    'data/tests',
    'frontend',
    'tests',
    'logs',
    'models',
    'results',
]:
    (PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)


def run_command(command, cwd=PROJECT_ROOT, env=None):
    print('$', ' '.join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)


if sys.version_info[:2] == (3, 10):
    eco_python = sys.executable
    eco_pip = shutil.which('pip') or str(Path(sys.executable).with_name('pip'))
    print('Using current Colab runtime:', eco_python)
else:
    conda_root = Path('/content/miniforge3')
    env_dir = conda_root / 'envs' / 'eco-rasa'
    installer = Path('/content/miniforge.sh')
    if not conda_root.exists():
        urllib.request.urlretrieve(
            'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
            installer,
        )
        run_command(['bash', str(installer), '-b', '-p', str(conda_root)], cwd=Path('/content'))
    conda_bin = conda_root / 'bin' / 'conda'
    if not env_dir.exists():
        run_command([str(conda_bin), 'create', '-y', '-n', 'eco-rasa', 'python=3.10'], cwd=Path('/content'))
    eco_python = str(env_dir / 'bin' / 'python')
    eco_pip = str(env_dir / 'bin' / 'pip')
    print('Using isolated Python 3.10 environment:', eco_python)

os.environ['ECO_PYTHON'] = eco_python
os.environ['ECO_PIP'] = eco_pip
os.environ['TMPDIR'] = '/content/rasa_tmp'
Path(os.environ['TMPDIR']).mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('ECO_PYTHON =', os.environ['ECO_PYTHON'])
print('ECO_PIP =', os.environ['ECO_PIP'])


$ bash /content/miniforge.sh -b -p /content/miniforge3
$ /content/miniforge3/bin/conda create -y -n eco-rasa python=3.10
Using isolated Python 3.10 environment: /content/miniforge3/envs/eco-rasa/bin/python
Project root: /content/eco_travel_advisor_project
ECO_PYTHON = /content/miniforge3/envs/eco-rasa/bin/python
ECO_PIP = /content/miniforge3/envs/eco-rasa/bin/pip


## Project Files


In [2]:
%%writefile .env.example
# Copy this file to .env and fill in real credentials only on your local machine.
# Do not commit .env to GitHub or submit it inside coursework archives.

CLIMATIQ_API_KEY=
AMADEUS_CLIENT_ID=
AMADEUS_CLIENT_SECRET=
RASA_REST_URL=http://localhost:5005/webhooks/rest/webhook


Writing .env.example


In [3]:
%%writefile requirements.txt
rasa>=3.6,<3.7
rasa-sdk>=3.6,<3.7
python-dotenv>=1.0.0
requests>=2.31.0
streamlit>=1.31.0
pytest>=7.4.0
pytest-mock>=3.12.0


Writing requirements.txt


In [4]:
%%writefile config.yml
version: "3.1"

language: en

pipeline:
- name: WhitespaceTokenizer
- name: RegexFeaturizer
- name: LexicalSyntacticFeaturizer
- name: CountVectorsFeaturizer
- name: CountVectorsFeaturizer
  analyzer: char_wb
  min_ngram: 1
  max_ngram: 4
- name: DIETClassifier
  epochs: 80
  constrain_similarities: true
- name: EntitySynonymMapper
- name: ResponseSelector
  epochs: 50
  constrain_similarities: true
- name: FallbackClassifier
  threshold: 0.35
  ambiguity_threshold: 0.10

policies:
- name: MemoizationPolicy
- name: RulePolicy
  core_fallback_threshold: 0.3
  core_fallback_action_name: "action_two_stage_clarification"
  enable_fallback_prediction: true
- name: UnexpecTEDIntentPolicy
  max_history: 5
  epochs: 80
- name: TEDPolicy
  max_history: 6
  epochs: 100
  constrain_similarities: true
assistant_id: 20260506-125101-flat-deposition


Writing config.yml


In [5]:
%%writefile credentials.yml
# Enable the REST channel for Streamlit and Rasa Webchat.
rest:

# Optional Socket.IO channel if students extend the project later.
socketio:
  user_message_evt: user_uttered
  bot_message_evt: bot_uttered
  session_persistence: true


Writing credentials.yml


In [6]:
%%writefile endpoints.yml
action_endpoint:
  url: "http://localhost:5055/webhook"


Writing endpoints.yml


In [7]:
%%writefile domain.yml
version: "3.1"

intents:
  - greet
  - goodbye
  - start_trip_planning
  - provide_destination
  - provide_origin
  - provide_dates
  - provide_budget
  - provide_sustainability_preference
  - provide_transport_mode
  - provide_accommodation_type
  - provide_user_location
  - ask_transport_options
  - ask_accommodation_options
  - ask_carbon_footprint
  - ask_cultural_experiences
  - ask_privacy_notice
  - ask_accessibility_support
  - request_human_handover
  - out_of_scope
  - thank_you
  - nlu_fallback

entities:
  - destination
  - origin
  - travel_date
  - return_date
  - budget
  - sustainability_level
  - transport_mode
  - accommodation_type
  - location

slots:
  destination:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: destination
  origin:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: origin
      - type: from_entity
        entity: location
  travel_date:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: travel_date
  return_date:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: return_date
  budget:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: budget
  sustainability_level:
    type: categorical
    values:
      - low
      - medium
      - high
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: sustainability_level
  transport_mode:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: transport_mode
  accommodation_type:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: accommodation_type
  user_location:
    type: text
    influence_conversation: false
    mappings:
      - type: from_entity
        entity: location
  carbon_score:
    type: float
    influence_conversation: false
    mappings:
      - type: custom
  selected_option:
    type: any
    influence_conversation: false
    mappings:
      - type: custom
  recommended_options:
    type: any
    influence_conversation: false
    mappings:
      - type: custom
  handover_required:
    type: bool
    influence_conversation: true
    initial_value: false
    mappings:
      - type: custom
  conversation_summary:
    type: any
    influence_conversation: false
    mappings:
      - type: custom
  clarification_attempts:
    type: float
    influence_conversation: true
    initial_value: 0
    mappings:
      - type: custom

responses:
  utter_greet:
    - text: "Hello. I am your Eco Travel Advisor. I can help plan a lower carbon trip with transport, accommodation, cultural activities, and offset options."
      buttons:
        - title: "Plan a sustainable trip"
          payload: "/start_trip_planning"
        - title: "Compare transport options"
          payload: "/ask_transport_options"
        - title: "Talk to a human advisor"
          payload: "/request_human_handover"
  utter_ask_destination:
    - text: "Which destination are you planning to visit?"
  utter_ask_origin:
    - text: "Where will you travel from?"
  utter_ask_dates:
    - text: "What are your travel dates? Please provide a departure date and a return date."
  utter_ask_budget:
    - text: "What is your approximate travel budget?"
  utter_ask_sustainability_level:
    - text: "How strong is your sustainability preference: low, medium, or high?"
      buttons:
        - title: "Low"
          payload: '/provide_sustainability_preference{"sustainability_level":"low"}'
        - title: "Medium"
          payload: '/provide_sustainability_preference{"sustainability_level":"medium"}'
        - title: "High"
          payload: '/provide_sustainability_preference{"sustainability_level":"high"}'
  utter_ask_transport_mode:
    - text: "Do you prefer train, bus, flight, car, walking, or cycling?"
      buttons:
        - title: "Train"
          payload: '/provide_transport_mode{"transport_mode":"train"}'
        - title: "Bus"
          payload: '/provide_transport_mode{"transport_mode":"bus"}'
        - title: "Flight"
          payload: '/provide_transport_mode{"transport_mode":"flight"}'
        - title: "Car"
          payload: '/provide_transport_mode{"transport_mode":"car"}'
  utter_goodbye:
    - text: "Goodbye. I hope your next trip is enjoyable, responsible, and lower carbon."
  utter_handover_started:
    - text: "I have prepared your conversation context for a human sustainable travel advisor."
  utter_privacy_notice:
    - text: "Privacy notice: this demo uses only the trip details you provide in the conversation. API keys must be stored in environment variables, not in source code. Do not enter sensitive personal data in this academic prototype."
  utter_accessibility_support:
    - text: "Accessibility support: the frontend uses readable labels, plain language, button alternatives, and colour labels that are also written as text for users who cannot rely on colour alone."
  utter_thank_you:
    - text: "You are welcome."
  utter_out_of_scope:
    - text: "I can help with sustainable travel planning. I cannot help with unrelated requests, but I can compare transport, hotels, carbon footprint, cultural activities, or arrange handover."

actions:
  - action_collect_trip_preferences
  - action_recommend_transport
  - action_calculate_carbon_footprint
  - action_recommend_hotels
  - action_recommend_cultural_experiences
  - action_rank_sustainable_options
  - action_package_handover_context
  - action_default_fallback
  - action_two_stage_clarification

session_config:
  session_expiration_time: 60
  carry_over_slots_to_new_session: true


Writing domain.yml


In [8]:
%%writefile data/nlu.yml
version: "3.1"

nlu:
  - intent: greet
    examples: |
      - hello
      - hi
      - hey
      - good morning
      - good evening
      - hello there
      - hi eco advisor
      - can you help me?
      - I need help with travel
      - hey, I want sustainable travel advice

  - intent: goodbye
    examples: |
      - goodbye
      - bye
      - see you later
      - thanks, bye
      - end the chat
      - talk to you later
      - that is all for now
      - I am done

  - intent: start_trip_planning
    examples: |
      - I want to plan a sustainable trip
      - help me plan an eco friendly holiday
      - plan a green trip for me
      - I need a sustainable tourism plan
      - can you help with an eco travel itinerary?
      - I want to organise a low carbon vacation
      - build me a responsible travel plan
      - I need an environmentally friendly trip
      - start planning my trip
      - let us plan a sustainable journey

  - intent: provide_destination
    examples: |
      - I want to visit [Amsterdam](destination)
      - My destination is [Barcelona](destination)
      - I am going to [Copenhagen](destination)
      - The trip is to [Lisbon](destination)
      - We are planning to travel to [Vienna](destination)
      - I would like recommendations for [Berlin](destination)
      - destination [Paris](destination)
      - it is [Prague](destination)
      - I am thinking about [Ljubljana](destination)
      - maybe [Stockholm](destination), but I am not sure

  - intent: provide_origin
    examples: |
      - I am travelling from [Berlin](origin)
      - My starting point is [Munich](origin)
      - I will depart from [London](origin)
      - origin is [Paris](origin)
      - I live in [Hamburg](origin)
      - We start in [Brussels](origin)
      - from [Frankfurt](origin)
      - leaving from [Madrid](origin)
      - I am currently in [Rome](origin)
      - our group starts from [Zurich](origin)

  - intent: provide_dates
    examples: |
      - I will travel on [12 June](travel_date) and return on [15 June](return_date)
      - from [1 July](travel_date) to [6 July](return_date)
      - departure is [next Friday](travel_date), return is [next Monday](return_date)
      - travel date [20 August](travel_date), return date [25 August](return_date)
      - I leave on [5 May 2026](travel_date) and come back on [9 May 2026](return_date)
      - between [10 September](travel_date) and [14 September](return_date)
      - my dates are [3 April](travel_date) until [7 April](return_date)
      - [18 October](travel_date) to [22 October](return_date)
      - I only know I want to go in [June](travel_date)
      - maybe [next month](travel_date), not sure about return

  - intent: provide_budget
    examples: |
      - my budget is [500](budget) euros
      - I can spend about [800](budget)
      - budget [1200](budget)
      - I have around [300](budget) EUR
      - keep it under [1000](budget)
      - maybe [700](budget) euros for the trip
      - I prefer a low budget around [250](budget)
      - maximum [1500](budget)
      - no more than [600](budget)
      - I am flexible, maybe [900](budget)

  - intent: provide_sustainability_preference
    examples: |
      - my sustainability level is [high](sustainability_level)
      - I want a [medium](sustainability_level) sustainability preference
      - make sustainability [low](sustainability_level)
      - I strongly prefer eco friendly options, [high](sustainability_level)
      - balanced sustainability is fine, [medium](sustainability_level)
      - I care mostly about price, [low](sustainability_level)
      - choose the greenest option, [high](sustainability_level)
      - I want a balanced approach, [medium](sustainability_level)
      - sustainability is important but not absolute, [medium](sustainability_level)
      - I want the lowest carbon choices, [high](sustainability_level)

  - intent: provide_transport_mode
    examples: |
      - I prefer [train](transport_mode)
      - use [bus](transport_mode) if possible
      - compare [flight](transport_mode)
      - I will travel by [car](transport_mode)
      - I want to go by [cycling](transport_mode)
      - [walking](transport_mode) is fine for local movement
      - choose [train](transport_mode) for the main journey
      - maybe [bus](transport_mode), maybe train
      - I need a [flight](transport_mode) because of time
      - [car](transport_mode) with two passengers

  - intent: provide_accommodation_type
    examples: |
      - I prefer an [eco hotel](accommodation_type)
      - find a [hostel](accommodation_type)
      - I want a [guesthouse](accommodation_type)
      - accommodation type should be [apartment](accommodation_type)
      - please look for [eco lodge](accommodation_type)
      - a [certified green hotel](accommodation_type) would be good
      - [budget hotel](accommodation_type)
      - [rural lodge](accommodation_type)
      - I need a [business hotel](accommodation_type)
      - maybe an [eco hostel](accommodation_type)

  - intent: provide_user_location
    examples: |
      - my current location is [Berlin](location)
      - I am currently in [Munich](location)
      - I live near [Paris](location)
      - user location [London](location)
      - I am based in [Brno](location)
      - I am in [Tunis](location)
      - start from my location in [Hamburg](location)
      - my city is [Stuttgart](location)

  - intent: ask_transport_options
    examples: |
      - show me low carbon transport options
      - compare train and flight emissions
      - what is the greenest transport?
      - recommend transport for this trip
      - which transport mode should I use?
      - give me transport recommendations
      - can I go by train?
      - is bus better than flying?
      - show sustainable mobility options
      - compare transport choices

  - intent: ask_accommodation_options
    examples: |
      - recommend eco friendly hotels
      - show green accommodation
      - find sustainable places to stay
      - I need an eco hotel
      - give me accommodation options
      - any certified green hotels?
      - where should I stay sustainably?
      - recommend a responsible hostel
      - show low impact accommodation
      - hotel options please

  - intent: ask_carbon_footprint
    examples: |
      - calculate my carbon footprint
      - estimate emissions for my trip
      - how much CO2 will this journey produce?
      - what is the carbon impact?
      - show emissions for train and flight
      - carbon footprint please
      - estimate CO2 for my travel mode
      - how green is this trip?
      - calculate transport emissions
      - tell me if this trip is low carbon

  - intent: ask_cultural_experiences
    examples: |
      - recommend cultural experiences
      - show local activities
      - find responsible cultural tours
      - what can I do with local communities?
      - suggest authentic experiences
      - eco friendly cultural activities
      - recommend museums and local markets
      - show sustainable tourism activities
      - local food tour recommendations
      - cultural experience options please

  - intent: ask_privacy_notice
    examples: |
      - what happens to my data?
      - explain privacy
      - do you store personal data?
      - show privacy notice
      - GDPR information please
      - how do you use my trip details?
      - is this conversation private?
      - privacy policy

  - intent: ask_accessibility_support
    examples: |
      - do you support accessibility?
      - accessibility options
      - can screen readers use this?
      - is the interface accessible?
      - I need accessible travel planning
      - explain accessibility support
      - can you avoid colour only labels?

  - intent: request_human_handover
    examples: |
      - I want to talk to a human advisor
      - connect me to a person
      - hand me over to a human
      - I need human help
      - can an advisor review this?
      - escalate this conversation
      - I do not understand, give me a person
      - transfer me to support
      - I want expert travel advice
      - please arrange human handover

  - intent: nlu_fallback
    examples: |
      - green place thing maybe not sure
      - trip stuff around there somehow
      - I want something nice, you know?
      - compare the thing with the other thing
      - maybe do the sustainable travel thing but different
      - not sure how to explain what I want

  - intent: thank_you
    examples: |
      - thank you
      - thanks
      - many thanks
      - great, thanks
      - that helps
      - appreciated
      - thanks for the recommendations
      - thank you very much

  - intent: out_of_scope
    examples: |
      - what is the weather on Mars?
      - write my biology essay
      - order pizza for me
      - book a medical appointment
      - what is the stock price of Tesla?
      - tell me a joke about politics
      - solve my tax problem
      - I want to buy a laptop
      - play music
      - explain quantum mechanics


Writing data/nlu.yml


In [9]:
%%writefile data/stories.yml
version: "3.1"

stories:
  - story: complete sustainable city break
    steps:
      - intent: greet
      - action: utter_greet
      - intent: start_trip_planning
      - action: utter_ask_destination
      - intent: provide_destination
        entities:
          - destination: Amsterdam
      - slot_was_set:
          - destination: Amsterdam
      - action: utter_ask_origin
      - intent: provide_origin
        entities:
          - origin: Berlin
      - slot_was_set:
          - origin: Berlin
      - action: utter_ask_dates
      - intent: provide_dates
        entities:
          - travel_date: 12 June
          - return_date: 15 June
      - slot_was_set:
          - travel_date: 12 June
          - return_date: 15 June
      - action: utter_ask_budget
      - intent: provide_budget
        entities:
          - budget: "700"
      - slot_was_set:
          - budget: "700"
      - action: utter_ask_sustainability_level
      - intent: provide_sustainability_preference
        entities:
          - sustainability_level: high
      - slot_was_set:
          - sustainability_level: high
      - action: utter_ask_transport_mode
      - intent: provide_transport_mode
        entities:
          - transport_mode: train
      - slot_was_set:
          - transport_mode: train
      - action: action_collect_trip_preferences
      - action: action_recommend_transport
      - action: action_recommend_hotels
      - action: action_recommend_cultural_experiences

  - story: rural eco tour
    steps:
      - intent: start_trip_planning
      - action: utter_ask_destination
      - intent: provide_destination
        entities:
          - destination: Ljubljana
      - slot_was_set:
          - destination: Ljubljana
      - action: utter_ask_origin
      - intent: provide_origin
        entities:
          - origin: Vienna
      - slot_was_set:
          - origin: Vienna
      - action: utter_ask_dates
      - intent: provide_dates
        entities:
          - travel_date: 4 August
          - return_date: 10 August
      - slot_was_set:
          - travel_date: 4 August
          - return_date: 10 August
      - action: utter_ask_budget
      - intent: provide_budget
        entities:
          - budget: "900"
      - slot_was_set:
          - budget: "900"
      - action: utter_ask_sustainability_level
      - intent: provide_sustainability_preference
        entities:
          - sustainability_level: high
      - slot_was_set:
          - sustainability_level: high
      - action: utter_ask_transport_mode
      - intent: provide_transport_mode
        entities:
          - transport_mode: train
      - slot_was_set:
          - transport_mode: train
      - action: action_collect_trip_preferences
      - action: action_recommend_transport
      - action: action_recommend_hotels
      - action: action_recommend_cultural_experiences

  - story: carbon conscious business trip
    steps:
      - intent: start_trip_planning
      - action: utter_ask_destination
      - intent: provide_destination
        entities:
          - destination: Copenhagen
      - slot_was_set:
          - destination: Copenhagen
      - action: utter_ask_origin
      - intent: provide_origin
        entities:
          - origin: London
      - slot_was_set:
          - origin: London
      - action: utter_ask_dates
      - intent: provide_dates
        entities:
          - travel_date: 8 September
          - return_date: 9 September
      - slot_was_set:
          - travel_date: 8 September
          - return_date: 9 September
      - action: utter_ask_budget
      - intent: provide_budget
        entities:
          - budget: "1200"
      - slot_was_set:
          - budget: "1200"
      - action: utter_ask_sustainability_level
      - intent: provide_sustainability_preference
        entities:
          - sustainability_level: medium
      - slot_was_set:
          - sustainability_level: medium
      - action: utter_ask_transport_mode
      - intent: provide_transport_mode
        entities:
          - transport_mode: train
      - slot_was_set:
          - transport_mode: train
      - action: action_collect_trip_preferences
      - action: action_recommend_transport
      - action: action_recommend_hotels
      - action: action_recommend_cultural_experiences
      - intent: ask_carbon_footprint
      - action: action_calculate_carbon_footprint

  - story: use current location as travel origin
    steps:
      - intent: start_trip_planning
      - action: utter_ask_destination
      - intent: provide_destination
        entities:
          - destination: Amsterdam
      - slot_was_set:
          - destination: Amsterdam
      - action: utter_ask_origin
      - intent: provide_user_location
        entities:
          - location: Berlin
      - slot_was_set:
          - origin: Berlin
          - user_location: Berlin
      - action: utter_ask_dates

  - story: accommodation recommendation
    steps:
      - intent: ask_accommodation_options
      - action: action_recommend_hotels
      - intent: provide_accommodation_type
        entities:
          - accommodation_type: eco hotel
      - slot_was_set:
          - accommodation_type: eco hotel
      - action: action_recommend_hotels

  - story: transport recommendation
    steps:
      - intent: ask_transport_options
      - action: action_recommend_transport
      - intent: provide_transport_mode
        entities:
          - transport_mode: bus
      - slot_was_set:
          - transport_mode: bus
      - action: action_recommend_transport

  - story: carbon footprint calculation
    steps:
      - intent: provide_transport_mode
        entities:
          - transport_mode: flight
      - slot_was_set:
          - transport_mode: flight
      - intent: ask_carbon_footprint
      - action: action_calculate_carbon_footprint

  - story: cultural experience recommendation
    steps:
      - intent: ask_cultural_experiences
      - action: action_recommend_cultural_experiences

  - story: user asks for human advisor
    steps:
      - intent: request_human_handover
      - action: action_package_handover_context
      - action: utter_handover_started

  - story: ambiguous message flow
    steps:
      - intent: nlu_fallback
      - action: action_two_stage_clarification

  - story: out of scope message flow
    steps:
      - intent: out_of_scope
      - action: utter_out_of_scope


Writing data/stories.yml


In [10]:
%%writefile data/rules.yml
version: "3.1"

rules:
  - rule: greet the user
    steps:
      - intent: greet
      - action: utter_greet

  - rule: say goodbye
    steps:
      - intent: goodbye
      - action: utter_goodbye

  - rule: start trip planning
    steps:
      - intent: start_trip_planning
      - action: utter_ask_destination

  - rule: capture destination and ask origin
    steps:
      - intent: provide_destination
      - action: utter_ask_origin

  - rule: capture origin and ask dates
    steps:
      - intent: provide_origin
      - action: utter_ask_dates

  - rule: capture current location and ask dates
    steps:
      - intent: provide_user_location
      - action: utter_ask_dates

  - rule: capture dates and ask budget
    steps:
      - intent: provide_dates
      - action: utter_ask_budget

  - rule: capture budget and ask sustainability preference
    steps:
      - intent: provide_budget
      - action: utter_ask_sustainability_level

  - rule: capture sustainability preference and ask transport mode
    steps:
      - intent: provide_sustainability_preference
      - action: utter_ask_transport_mode

  - rule: ask privacy notice
    steps:
      - intent: ask_privacy_notice
      - action: utter_privacy_notice

  - rule: ask accessibility support
    steps:
      - intent: ask_accessibility_support
      - action: utter_accessibility_support

  - rule: thank user
    steps:
      - intent: thank_you
      - action: utter_thank_you

  - rule: out of scope response
    steps:
      - intent: out_of_scope
      - action: utter_out_of_scope

  - rule: recommend transport on request
    steps:
      - intent: ask_transport_options
      - action: action_recommend_transport

  - rule: recommend accommodation on request
    steps:
      - intent: ask_accommodation_options
      - action: action_recommend_hotels

  - rule: calculate carbon footprint on request
    steps:
      - intent: ask_carbon_footprint
      - action: action_calculate_carbon_footprint

  - rule: recommend cultural experiences on request
    steps:
      - intent: ask_cultural_experiences
      - action: action_recommend_cultural_experiences

  - rule: human handover
    steps:
      - intent: request_human_handover
      - action: action_package_handover_context
      - action: utter_handover_started

  - rule: fallback two stage clarification
    steps:
      - intent: nlu_fallback
      - action: action_two_stage_clarification


Writing data/rules.yml


In [11]:
%%writefile data/tests/test_stories.yml
version: "3.1"

stories:
  - story: test sustainable trip path
    steps:
      - user: |
          hello
        intent: greet
      - action: utter_greet
      - user: |
          I want to plan a sustainable trip
        intent: start_trip_planning
      - action: utter_ask_destination
      - user: |
          I want to visit [Amsterdam](destination)
        intent: provide_destination
      - slot_was_set:
          - destination: Amsterdam
      - action: utter_ask_origin
      - user: |
          I am travelling from [Berlin](origin)
        intent: provide_origin
      - slot_was_set:
          - origin: Berlin
      - action: utter_ask_dates
      - user: |
          from [12 June](travel_date) to [15 June](return_date)
        intent: provide_dates
      - slot_was_set:
          - travel_date: 12 June
          - return_date: 15 June
      - action: utter_ask_budget
      - user: |
          my budget is [700](budget) euros
        intent: provide_budget
      - slot_was_set:
          - budget: "700"
      - action: utter_ask_sustainability_level
      - user: |
          [medium](sustainability_level)
        intent: provide_sustainability_preference
      - slot_was_set:
          - sustainability_level: medium
      - action: utter_ask_transport_mode
      - user: |
          [train](transport_mode)
        intent: provide_transport_mode
      - slot_was_set:
          - transport_mode: train
      - action: action_collect_trip_preferences
      - action: action_recommend_transport
      - action: action_recommend_hotels
      - action: action_recommend_cultural_experiences

  - story: test fallback path
    steps:
      - user: |
          strange unclear message qwerty
        intent: nlu_fallback
      - action: action_two_stage_clarification

  - story: test handover path
    steps:
      - user: |
          connect me to a person
        intent: request_human_handover
      - action: action_package_handover_context
      - action: utter_handover_started


Writing data/tests/test_stories.yml


In [12]:
%%writefile actions/__init__.py


Writing actions/__init__.py


In [13]:
%%writefile actions/actions.py
"""Custom actions for the Eco Travel Advisor Rasa assistant.

The file is intentionally written in a beginner friendly style. It separates API
clients, recommendation logic, and Rasa action classes so that students can test
and extend each part independently.
"""

from __future__ import annotations

import json
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Text

import requests
from dotenv import load_dotenv
from rasa_sdk import Action, Tracker
from rasa_sdk.executor import CollectingDispatcher
from rasa_sdk.events import FollowupAction, SlotSet

load_dotenv()
logger = logging.getLogger(__name__)

MOCK_DATA_DIR = Path(__file__).resolve().parent / "mock_data"


def load_mock_json(filename: str) -> List[Dict[str, Any]]:
    """Load local JSON mock data from actions/mock_data."""
    path = MOCK_DATA_DIR / filename
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def safe_float(value: Any, default: float = 0.0) -> float:
    """Convert text such as '700 euros' to a float where possible."""
    if value is None:
        return default
    if isinstance(value, (int, float)):
        return float(value)
    digits = "".join(ch for ch in str(value) if ch.isdigit() or ch == ".")
    try:
        return float(digits) if digits else default
    except ValueError:
        return default


def normalise_text(value: Optional[Any]) -> str:
    return str(value or "").strip().lower()


class CarbonImpactClassifier:
    """Classifies carbon values into simple labels for the UI."""

    @staticmethod
    def classify(carbon_kg: float) -> str:
        if carbon_kg <= 25:
            return "green"
        if carbon_kg <= 100:
            return "amber"
        return "red"


class ClimatiqClient:
    """Minimal Climatiq integration wrapper with robust local fallback.

    This project keeps the real API call small and defensive because the
    assistant must remain functional even when the API key is missing, a timeout
    occurs, or the external response shape changes.
    """

    BASE_URL = "https://api.climatiq.io/estimate"

    ACTIVITY_IDS = {
        "train": "passenger_train-route_type_na-fuel_source_na",
        "bus": "passenger_vehicle-vehicle_type_bus-fuel_source_na",
        "flight": "passenger_flight-route_type_domestic-aircraft_type_na-distance_na-class_na-rf_included",
        "car": "passenger_vehicle-vehicle_type_car-fuel_source_na-engine_size_na",
    }

    MOCK_FACTORS_KG_PER_KM = {
        "train": 0.041,
        "bus": 0.027,
        "flight": 0.255,
        "car": 0.171,
        "walking": 0.0,
        "cycling": 0.0,
    }

    MOCK_DISTANCES_KM = {
        ("berlin", "amsterdam"): 655,
        ("london", "copenhagen"): 955,
        ("vienna", "ljubljana"): 385,
        ("munich", "vienna"): 435,
        ("paris", "barcelona"): 1035,
    }

    def __init__(self, api_key: Optional[str] = None, timeout_seconds: int = 8):
        self.api_key = api_key or os.getenv("CLIMATIQ_API_KEY")
        self.timeout_seconds = timeout_seconds

    def estimate_transport(
        self,
        mode: str,
        origin: Optional[str] = None,
        destination: Optional[str] = None,
        distance_km: Optional[float] = None,
        passengers: int = 1,
    ) -> Dict[str, Any]:
        mode_key = normalise_text(mode) or "train"
        distance = distance_km or self.mock_distance_km(origin, destination)

        if mode_key in {"walking", "cycling"}:
            return self.mock_estimate(mode_key, distance, passengers, reason="zero direct emissions mode")

        if not self.api_key:
            return self.mock_estimate(mode_key, distance, passengers, reason="missing CLIMATIQ_API_KEY")

        activity_id = self.ACTIVITY_IDS.get(mode_key)
        if not activity_id:
            return self.mock_estimate(mode_key, distance, passengers, reason="unsupported mode for Climatiq")

        payload = {
            "emission_factor": {
                "activity_id": activity_id,
                "data_version": "^21",
            },
            "parameters": {
                "distance": distance,
                "distance_unit": "km",
            },
        }
        headers = {"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"}

        try:
            response = requests.post(self.BASE_URL, json=payload, headers=headers, timeout=self.timeout_seconds)
            if response.status_code in {401, 403, 429}:
                return self.mock_estimate(mode_key, distance, passengers, reason=f"Climatiq status {response.status_code}")
            response.raise_for_status()
            data = response.json()
            co2e = float(data.get("co2e", 0)) * max(passengers, 1)
            if co2e <= 0:
                return self.mock_estimate(mode_key, distance, passengers, reason="invalid Climatiq response")
            return {
                "mode": mode_key,
                "carbon_kg": round(co2e, 2),
                "distance_km": round(distance, 1),
                "source": "climatiq_api",
                "fallback_reason": None,
            }
        except requests.Timeout:
            return self.mock_estimate(mode_key, distance, passengers, reason="Climatiq timeout")
        except requests.RequestException as exc:
            logger.warning("Climatiq request failed: %s", exc)
            return self.mock_estimate(mode_key, distance, passengers, reason="Climatiq request error")
        except (ValueError, TypeError) as exc:
            logger.warning("Climatiq response parsing failed: %s", exc)
            return self.mock_estimate(mode_key, distance, passengers, reason="Climatiq invalid response")

    def mock_distance_km(self, origin: Optional[str], destination: Optional[str]) -> float:
        key = (normalise_text(origin), normalise_text(destination))
        reverse_key = (key[1], key[0])
        return float(self.MOCK_DISTANCES_KM.get(key) or self.MOCK_DISTANCES_KM.get(reverse_key) or 500.0)

    def mock_estimate(self, mode: str, distance_km: float, passengers: int, reason: str) -> Dict[str, Any]:
        factor = self.MOCK_FACTORS_KG_PER_KM.get(mode, self.MOCK_FACTORS_KG_PER_KM["train"])
        carbon_kg = distance_km * factor * max(passengers, 1)
        return {
            "mode": mode,
            "carbon_kg": round(carbon_kg, 2),
            "distance_km": round(distance_km, 1),
            "source": "mock_emission_factor",
            "fallback_reason": reason,
        }


class AmadeusClient:
    """Amadeus sandbox integration wrapper with mock fallback.

    The methods demonstrate correct credential handling and failure behaviour.
    Students can replace the placeholder search logic with richer Amadeus
    endpoints when they receive valid sandbox credentials.
    """

    TOKEN_URL = "https://test.api.amadeus.com/v1/security/oauth2/token"
    HOTEL_SEARCH_URL = "https://test.api.amadeus.com/v3/shopping/hotel-offers"
    FLIGHT_SEARCH_URL = "https://test.api.amadeus.com/v2/shopping/flight-offers"

    def __init__(self, client_id: Optional[str] = None, client_secret: Optional[str] = None, timeout_seconds: int = 8):
        self.client_id = client_id or os.getenv("AMADEUS_CLIENT_ID")
        self.client_secret = client_secret or os.getenv("AMADEUS_CLIENT_SECRET")
        self.timeout_seconds = timeout_seconds
        self._access_token: Optional[str] = None

    def has_credentials(self) -> bool:
        return bool(self.client_id and self.client_secret)

    def get_access_token(self) -> Optional[str]:
        if not self.has_credentials():
            logger.info("Amadeus credentials missing. Using mock travel data.")
            return None
        if self._access_token:
            return self._access_token

        try:
            response = requests.post(
                self.TOKEN_URL,
                data={
                    "grant_type": "client_credentials",
                    "client_id": self.client_id,
                    "client_secret": self.client_secret,
                },
                timeout=self.timeout_seconds,
            )
            response.raise_for_status()
            token = response.json().get("access_token")
            self._access_token = token
            return token
        except requests.RequestException as exc:
            logger.warning("Amadeus token request failed: %s", exc)
            return None

    def search_hotels(self, destination: Optional[str]) -> List[Dict[str, Any]]:
        token = self.get_access_token()
        if not token:
            return self.mock_hotels(destination)

        # Placeholder for a real Amadeus hotel search. The result is normalised
        # to the same local schema so that the recommendation engine stays stable.
        try:
            response = requests.get(
                self.HOTEL_SEARCH_URL,
                headers={"Authorization": f"Bearer {token}"},
                params={"cityCode": self.city_code(destination), "adults": 1},
                timeout=self.timeout_seconds,
            )
            response.raise_for_status()
            raw_items = response.json().get("data", [])
            if not raw_items:
                return self.mock_hotels(destination)
            return self.normalise_hotels(raw_items, destination)
        except requests.RequestException as exc:
            logger.warning("Amadeus hotel search failed: %s", exc)
            return self.mock_hotels(destination)

    def search_flights(self, origin: Optional[str], destination: Optional[str]) -> List[Dict[str, Any]]:
        token = self.get_access_token()
        if not token:
            return load_mock_json("transport_options.json")

        try:
            response = requests.get(
                self.FLIGHT_SEARCH_URL,
                headers={"Authorization": f"Bearer {token}"},
                params={
                    "originLocationCode": self.city_code(origin),
                    "destinationLocationCode": self.city_code(destination),
                    "departureDate": "2026-06-12",
                    "adults": 1,
                    "max": 3,
                },
                timeout=self.timeout_seconds,
            )
            response.raise_for_status()
            if not response.json().get("data"):
                return load_mock_json("transport_options.json")
            return load_mock_json("transport_options.json")
        except requests.RequestException as exc:
            logger.warning("Amadeus flight search failed: %s", exc)
            return load_mock_json("transport_options.json")

    def mock_hotels(self, destination: Optional[str]) -> List[Dict[str, Any]]:
        hotels = load_mock_json("hotels.json")
        dest = normalise_text(destination)
        selected = [h for h in hotels if normalise_text(h.get("destination")) in {dest, "any"}]
        return selected or hotels

    def normalise_hotels(self, raw_items: List[Dict[str, Any]], destination: Optional[str]) -> List[Dict[str, Any]]:
        normalised = []
        for idx, item in enumerate(raw_items[:5], start=1):
            hotel = item.get("hotel", {})
            offer = (item.get("offers") or [{}])[0]
            price = safe_float((offer.get("price") or {}).get("total"), 150.0)
            normalised.append(
                {
                    "id": f"amadeus_hotel_{idx}",
                    "name": hotel.get("name", f"Amadeus Hotel {idx}"),
                    "destination": destination or "Unknown",
                    "type": "hotel",
                    "nightly_price": price,
                    "eco_rating": 70,
                    "carbon_kg_per_night": 12,
                    "certifications": ["API result, sustainability not verified"],
                    "relevance": 0.70,
                    "description": "Hotel returned by Amadeus sandbox. Sustainability fields are illustrative placeholders.",
                }
            )
        return normalised

    def city_code(self, city: Optional[str]) -> str:
        mapping = {
            "amsterdam": "AMS",
            "berlin": "BER",
            "copenhagen": "CPH",
            "london": "LON",
            "ljubljana": "LJU",
            "vienna": "VIE",
            "paris": "PAR",
            "barcelona": "BCN",
        }
        return mapping.get(normalise_text(city), "AMS")


class RecommendationEngine:
    """Ranks travel options according to carbon, price, and user preference."""

    WEIGHTS = {
        "low": {"carbon": 0.20, "price": 0.55, "preference": 0.25},
        "medium": {"carbon": 0.35, "price": 0.35, "preference": 0.30},
        "high": {"carbon": 0.55, "price": 0.20, "preference": 0.25},
    }

    @classmethod
    def weights_for(cls, sustainability_level: Optional[str]) -> Dict[str, float]:
        return cls.WEIGHTS.get(normalise_text(sustainability_level), cls.WEIGHTS["medium"])

    @staticmethod
    def carbon_score(carbon_kg: float) -> float:
        # Lower emissions should produce a higher score.
        return max(0.0, min(100.0, 100.0 - carbon_kg))

    @staticmethod
    def price_score(price: float, budget: Optional[Any]) -> float:
        budget_value = safe_float(budget, default=800.0) or 800.0
        if price <= budget_value:
            return max(0.0, 100.0 - (price / budget_value) * 35.0)
        overspend_ratio = min(price / budget_value, 3.0)
        return max(0.0, 65.0 - (overspend_ratio - 1.0) * 45.0)

    @staticmethod
    def preference_score(option: Dict[str, Any]) -> float:
        if "eco_rating" in option:
            return float(option.get("eco_rating", 70))
        if "community_benefit" in option:
            return float(option.get("community_benefit", 70))
        return float(option.get("relevance", 0.7)) * 100.0

    @classmethod
    def rank_options(
        cls,
        options: List[Dict[str, Any]],
        sustainability_level: Optional[str],
        budget: Optional[Any],
        option_type: str,
    ) -> List[Dict[str, Any]]:
        weights = cls.weights_for(sustainability_level)
        ranked = []
        for option in options:
            price = float(option.get("price") or option.get("nightly_price") or 0)
            carbon = float(option.get("carbon_kg") or option.get("carbon_kg_per_night") or 0)
            score = (
                weights["carbon"] * cls.carbon_score(carbon)
                + weights["price"] * cls.price_score(price, budget)
                + weights["preference"] * cls.preference_score(option)
            )
            card = {
                "id": option.get("id"),
                "type": option_type,
                "title": option.get("display_name") or option.get("name") or "Recommendation",
                "subtitle": option.get("type") or option.get("mode") or option.get("category") or option_type,
                "price_eur": round(price, 2),
                "carbon_kg": round(carbon, 2),
                "carbon_label": CarbonImpactClassifier.classify(carbon),
                "score": round(score, 2),
                "description": option.get("description", ""),
                "details": option,
            }
            ranked.append(card)
        return sorted(ranked, key=lambda item: item["score"], reverse=True)


def build_context_summary(tracker: Tracker, reason: str) -> Dict[str, Any]:
    """Build a readable handover dictionary for a human advisor."""
    latest_message = tracker.latest_message.get("text") if tracker.latest_message else None
    return {
        "destination": tracker.get_slot("destination"),
        "origin": tracker.get_slot("origin"),
        "travel_date": tracker.get_slot("travel_date"),
        "return_date": tracker.get_slot("return_date"),
        "budget": tracker.get_slot("budget"),
        "sustainability_level": tracker.get_slot("sustainability_level"),
        "preferred_transport": tracker.get_slot("transport_mode"),
        "accommodation_type": tracker.get_slot("accommodation_type"),
        "recommended_options_already_shown": tracker.get_slot("recommended_options"),
        "unresolved_request": latest_message,
        "latest_user_message": latest_message,
        "reason_for_escalation": reason,
    }


class ActionCollectTripPreferences(Action):
    def name(self) -> Text:
        return "action_collect_trip_preferences"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        required_slots = [
            ("destination", "Which destination are you planning to visit?"),
            ("origin", "Where will you travel from?"),
            ("travel_date", "What is your departure date?"),
            ("return_date", "What is your return date?"),
            ("budget", "What is your approximate budget?"),
            ("sustainability_level", "Should sustainability preference be low, medium, or high?"),
        ]
        for slot_name, question in required_slots:
            if not tracker.get_slot(slot_name):
                dispatcher.utter_message(text=question)
                return []

        dispatcher.utter_message(text="Thank you. I have enough information to prepare sustainable recommendations.")
        summary = (
            f"Trip summary: {tracker.get_slot('origin')} to {tracker.get_slot('destination')}, "
            f"from {tracker.get_slot('travel_date')} to {tracker.get_slot('return_date')}, "
            f"budget {tracker.get_slot('budget')}, sustainability preference {tracker.get_slot('sustainability_level')}."
        )
        dispatcher.utter_message(text=summary)
        return []


class ActionRecommendTransport(Action):
    def name(self) -> Text:
        return "action_recommend_transport"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        origin = tracker.get_slot("origin")
        destination = tracker.get_slot("destination")
        budget = tracker.get_slot("budget")
        sustainability_level = tracker.get_slot("sustainability_level")
        requested_mode = normalise_text(tracker.get_slot("transport_mode"))

        transport_options = load_mock_json("transport_options.json")
        if requested_mode:
            transport_options = [opt for opt in transport_options if normalise_text(opt.get("mode")) == requested_mode] or transport_options

        climatiq = ClimatiqClient()
        enriched_options = []
        for option in transport_options:
            estimate = climatiq.estimate_transport(option.get("mode", "train"), origin=origin, destination=destination)
            enriched = dict(option)
            enriched["carbon_kg"] = estimate["carbon_kg"]
            enriched["carbon_source"] = estimate["source"]
            enriched["fallback_reason"] = estimate["fallback_reason"]
            enriched_options.append(enriched)

        cards = RecommendationEngine.rank_options(enriched_options, sustainability_level, budget, "transport")[:3]
        dispatcher.utter_message(
            text="Here are the best transport options ranked by carbon impact, price, and your sustainability preference.",
            json_message={"cards": cards},
        )
        return [SlotSet("recommended_options", {"transport": cards}), SlotSet("selected_option", cards[0] if cards else None)]


class ActionCalculateCarbonFootprint(Action):
    def name(self) -> Text:
        return "action_calculate_carbon_footprint"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        mode = tracker.get_slot("transport_mode") or "train"
        origin = tracker.get_slot("origin")
        destination = tracker.get_slot("destination")
        estimate = ClimatiqClient().estimate_transport(mode, origin=origin, destination=destination)
        label = CarbonImpactClassifier.classify(estimate["carbon_kg"])
        fallback_note = f" Fallback reason: {estimate['fallback_reason']}." if estimate.get("fallback_reason") else ""
        dispatcher.utter_message(
            text=(
                f"Estimated carbon footprint for {estimate['mode']}: {estimate['carbon_kg']} kg CO2e "
                f"for approximately {estimate['distance_km']} km. Carbon label: {label}.{fallback_note}"
            ),
            json_message={"carbon_label": label, "estimate": estimate},
        )
        return [SlotSet("carbon_score", estimate["carbon_kg"])]


class ActionRecommendHotels(Action):
    def name(self) -> Text:
        return "action_recommend_hotels"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        destination = tracker.get_slot("destination")
        accommodation_type = normalise_text(tracker.get_slot("accommodation_type"))
        budget = tracker.get_slot("budget")
        sustainability_level = tracker.get_slot("sustainability_level")

        hotels = AmadeusClient().search_hotels(destination)
        if accommodation_type:
            filtered = [h for h in hotels if accommodation_type in normalise_text(h.get("type"))]
            hotels = filtered or hotels

        cards = RecommendationEngine.rank_options(hotels, sustainability_level, budget, "accommodation")[:3]
        dispatcher.utter_message(
            text="Here are sustainable accommodation recommendations.",
            json_message={"cards": cards},
        )
        return [SlotSet("recommended_options", {"accommodation": cards}), SlotSet("selected_option", cards[0] if cards else None)]


class ActionRecommendCulturalExperiences(Action):
    def name(self) -> Text:
        return "action_recommend_cultural_experiences"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        destination = normalise_text(tracker.get_slot("destination"))
        budget = tracker.get_slot("budget")
        sustainability_level = tracker.get_slot("sustainability_level")
        experiences = load_mock_json("cultural_experiences.json")
        selected = [e for e in experiences if normalise_text(e.get("destination")) in {destination, "any"}]
        if not selected:
            selected = experiences
        cards = RecommendationEngine.rank_options(selected, sustainability_level, budget, "cultural_experience")[:3]
        dispatcher.utter_message(
            text="Here are responsible cultural experiences that support local value creation.",
            json_message={"cards": cards},
        )
        return [SlotSet("recommended_options", {"cultural_experiences": cards}), SlotSet("selected_option", cards[0] if cards else None)]


class ActionRankSustainableOptions(Action):
    def name(self) -> Text:
        return "action_rank_sustainable_options"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        budget = tracker.get_slot("budget")
        sustainability_level = tracker.get_slot("sustainability_level")
        options = load_mock_json("transport_options.json") + AmadeusClient().mock_hotels(tracker.get_slot("destination"))
        cards = RecommendationEngine.rank_options(options, sustainability_level, budget, "mixed")[:5]
        dispatcher.utter_message(
            text="I ranked the available sustainable options using carbon, price, and preference fit.",
            json_message={"cards": cards},
        )
        return [SlotSet("recommended_options", {"mixed": cards})]


class ActionPackageHandoverContext(Action):
    def name(self) -> Text:
        return "action_package_handover_context"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        summary = build_context_summary(tracker, reason="User requested human handover or clarification failed")
        dispatcher.utter_message(
            text="I have packaged your trip context for a human advisor.",
            json_message={"handover_required": True, "handover_context": summary},
        )
        return [SlotSet("handover_required", True), SlotSet("conversation_summary", summary)]


class ActionDefaultFallback(Action):
    def name(self) -> Text:
        return "action_default_fallback"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        dispatcher.utter_message(
            text="I did not understand that clearly. Please ask about transport, hotels, carbon footprint, cultural experiences, or human handover."
        )
        return [SlotSet("clarification_attempts", 1)]


class ActionTwoStageClarification(Action):
    def name(self) -> Text:
        return "action_two_stage_clarification"

    def run(self, dispatcher: CollectingDispatcher, tracker: Tracker, domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
        attempts = int(tracker.get_slot("clarification_attempts") or 0)
        if attempts < 1:
            dispatcher.utter_message(
                text="I am not fully sure what you need. Please choose one option: transport, hotels, carbon footprint, cultural activities, or human advisor.",
                buttons=[
                    {"title": "Transport", "payload": "/ask_transport_options"},
                    {"title": "Hotels", "payload": "/ask_accommodation_options"},
                    {"title": "Carbon footprint", "payload": "/ask_carbon_footprint"},
                    {"title": "Cultural experiences", "payload": "/ask_cultural_experiences"},
                    {"title": "Human advisor", "payload": "/request_human_handover"},
                ],
            )
            return [SlotSet("clarification_attempts", 1)]

        dispatcher.utter_message(text="I am still not confident about your request, so I will prepare a human handover context.")
        return [
            SlotSet("clarification_attempts", 0),
            SlotSet("handover_required", True),
            FollowupAction("action_package_handover_context"),
        ]


Writing actions/actions.py


In [14]:
%%writefile actions/mock_data/transport_options.json
[
  {
    "id": "transport_train",
    "mode": "train",
    "display_name": "Intercity train",
    "price": 95,
    "carbon_kg": 19.7,
    "duration_hours": 6.5,
    "relevance": 0.95,
    "description": "Usually the best balance between low emissions, comfort, and city centre access."
  },
  {
    "id": "transport_bus",
    "mode": "bus",
    "display_name": "Long distance coach",
    "price": 45,
    "carbon_kg": 13.0,
    "duration_hours": 8.0,
    "relevance": 0.88,
    "description": "Low cost and low carbon, but often slower than rail."
  },
  {
    "id": "transport_flight",
    "mode": "flight",
    "display_name": "Short haul flight",
    "price": 130,
    "carbon_kg": 122.4,
    "duration_hours": 2.0,
    "relevance": 0.7,
    "description": "Fast, but normally has the highest carbon impact for short city trips."
  },
  {
    "id": "transport_car",
    "mode": "car",
    "display_name": "Private car",
    "price": 115,
    "carbon_kg": 82.1,
    "duration_hours": 7.0,
    "relevance": 0.72,
    "description": "Flexible, but emissions depend heavily on vehicle type and occupancy."
  },
  {
    "id": "transport_cycling",
    "mode": "cycling",
    "display_name": "Cycling for local movement",
    "price": 20,
    "carbon_kg": 0.0,
    "duration_hours": 1.0,
    "relevance": 0.85,
    "description": "Best for local journeys and last mile mobility, not suitable for every intercity route."
  }
]


Writing actions/mock_data/transport_options.json


In [15]:
%%writefile actions/mock_data/hotels.json
[
  {
    "id": "hotel_ams_001",
    "name": "Canal Green Stay",
    "destination": "Amsterdam",
    "type": "eco hotel",
    "nightly_price": 145,
    "eco_rating": 92,
    "carbon_kg_per_night": 7.5,
    "certifications": [
      "Green Key",
      "renewable energy"
    ],
    "relevance": 0.96,
    "description": "Certified eco hotel close to tram lines, bike rental, and local food markets."
  },
  {
    "id": "hotel_ams_002",
    "name": "Budget Bike Hostel",
    "destination": "Amsterdam",
    "type": "hostel",
    "nightly_price": 62,
    "eco_rating": 78,
    "carbon_kg_per_night": 5.2,
    "certifications": [
      "waste reduction",
      "bike friendly"
    ],
    "relevance": 0.87,
    "description": "Affordable hostel with bike facilities and low water consumption measures."
  },
  {
    "id": "hotel_cph_001",
    "name": "Nordic Zero Hotel",
    "destination": "Copenhagen",
    "type": "business hotel",
    "nightly_price": 185,
    "eco_rating": 90,
    "carbon_kg_per_night": 8.9,
    "certifications": [
      "Green Key",
      "carbon reporting"
    ],
    "relevance": 0.91,
    "description": "Business friendly hotel with renewable energy sourcing and public transport access."
  },
  {
    "id": "hotel_lju_001",
    "name": "Forest Edge Eco Lodge",
    "destination": "Ljubljana",
    "type": "eco lodge",
    "nightly_price": 110,
    "eco_rating": 95,
    "carbon_kg_per_night": 4.8,
    "certifications": [
      "local sourcing",
      "nature conservation"
    ],
    "relevance": 0.94,
    "description": "Small rural lodge supporting local producers and protected area education."
  },
  {
    "id": "hotel_global_001",
    "name": "Urban Responsible Guesthouse",
    "destination": "Any",
    "type": "guesthouse",
    "nightly_price": 88,
    "eco_rating": 82,
    "carbon_kg_per_night": 6.4,
    "certifications": [
      "energy efficient",
      "local employment"
    ],
    "relevance": 0.8,
    "description": "Mock fallback guesthouse for destinations not found in the local database."
  }
]


Writing actions/mock_data/hotels.json


In [16]:
%%writefile actions/mock_data/cultural_experiences.json
[
  {
    "id": "exp_ams_001",
    "name": "Local Food Market Walk",
    "destination": "Amsterdam",
    "category": "food culture",
    "price": 35,
    "carbon_kg": 1.2,
    "community_benefit": 90,
    "relevance": 0.93,
    "description": "Small group walking tour focused on local producers and seasonal food."
  },
  {
    "id": "exp_ams_002",
    "name": "Canal Heritage by Electric Boat",
    "destination": "Amsterdam",
    "category": "heritage",
    "price": 42,
    "carbon_kg": 2.6,
    "community_benefit": 76,
    "relevance": 0.86,
    "description": "Low emission cultural route explaining water management and local history."
  },
  {
    "id": "exp_lju_001",
    "name": "Bee Keeping Heritage Visit",
    "destination": "Ljubljana",
    "category": "rural culture",
    "price": 55,
    "carbon_kg": 1.8,
    "community_benefit": 95,
    "relevance": 0.92,
    "description": "Community run experience showing Slovenian beekeeping traditions and biodiversity."
  },
  {
    "id": "exp_cph_001",
    "name": "Copenhagen Cycling Architecture Tour",
    "destination": "Copenhagen",
    "category": "urban sustainability",
    "price": 48,
    "carbon_kg": 0.4,
    "community_benefit": 80,
    "relevance": 0.89,
    "description": "Guided bicycle tour explaining cycling infrastructure and sustainable urban design."
  },
  {
    "id": "exp_global_001",
    "name": "Responsible Local Walking Tour",
    "destination": "Any",
    "category": "local culture",
    "price": 30,
    "carbon_kg": 0.6,
    "community_benefit": 82,
    "relevance": 0.78,
    "description": "Mock fallback walking tour supporting local guides."
  }
]


Writing actions/mock_data/cultural_experiences.json


In [17]:
%%writefile actions/mock_data/carbon_offsets.json
[
  {
    "id": "offset_001",
    "name": "Verified Reforestation Portfolio",
    "price_per_tonne_eur": 18,
    "standard": "VCS style mock standard",
    "description": "Mock offset option. Used only for demonstration, not a real purchasing link."
  },
  {
    "id": "offset_002",
    "name": "Renewable Energy Community Fund",
    "price_per_tonne_eur": 22,
    "standard": "Gold Standard style mock standard",
    "description": "Mock renewable energy offset supporting community energy projects."
  },
  {
    "id": "offset_003",
    "name": "Peatland Restoration Fund",
    "price_per_tonne_eur": 27,
    "standard": "Biodiversity co benefit mock standard",
    "description": "Mock nature based option with biodiversity and water retention benefits."
  }
]


Writing actions/mock_data/carbon_offsets.json


In [18]:
%%writefile frontend/streamlit_app.py
"""Simple Streamlit frontend for the Eco Travel Advisor Rasa bot."""

import json
import os
import uuid
from datetime import date, timedelta
from typing import Any, Dict, List, Optional

import requests
import streamlit as st
from dotenv import load_dotenv

load_dotenv()

RASA_REST_URL = os.getenv("RASA_REST_URL", "http://localhost:5005/webhooks/rest/webhook")

st.set_page_config(page_title="Eco Travel Advisor", layout="centered")
st.title("Eco Travel Advisor")
st.caption("Sustainable tourism planning with Rasa, custom actions, carbon labels, and human handover support.")


def send_to_rasa(sender_id: str, message: str) -> List[Dict[str, Any]]:
    payload = {"sender": sender_id, "message": message}
    try:
        response = requests.post(RASA_REST_URL, json=payload, timeout=15)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        return [{"text": f"Could not reach Rasa server at {RASA_REST_URL}. Error: {exc}"}]


def label_badge(label: str) -> str:
    label = (label or "unknown").lower()
    return {
        "green": "[GREEN] LOW IMPACT",
        "amber": "[AMBER] MEDIUM IMPACT",
        "red": "[RED] HIGH IMPACT",
    }.get(label, "[UNKNOWN] IMPACT")


def append_assistant_response(response: Dict[str, Any]) -> None:
    st.session_state.messages.append(
        {
            "role": "assistant",
            "text": response.get("text"),
            "custom": response.get("custom") or {},
            "buttons": response.get("buttons") or [],
        }
    )


def bootstrap_conversation() -> None:
    for response in send_to_rasa(st.session_state.sender_id, "/greet"):
        append_assistant_response(response)
    st.session_state.bootstrapped = True


def reset_conversation() -> None:
    st.session_state.sender_id = f"streamlit_{uuid.uuid4().hex}"
    st.session_state.messages = []
    st.session_state.pending_user_message = None
    st.session_state.bootstrapped = False


def queue_user_message(raw_message: str, display_message: Optional[str] = None) -> None:
    st.session_state.pending_user_message = {
        "raw": raw_message,
        "display": display_message if display_message is not None else raw_message,
    }


def handle_user_message(raw_message: str, display_message: Optional[str] = None) -> None:
    st.session_state.messages.append(
        {
            "role": "user",
            "text": display_message if display_message is not None else raw_message,
        }
    )

    for response in send_to_rasa(st.session_state.sender_id, raw_message):
        append_assistant_response(response)


def build_intent_payload(intent_name: str, entities: Dict[str, Any]) -> str:
    return f"/{intent_name}{json.dumps(entities, ensure_ascii=True)}"


def get_latest_assistant_message() -> Optional[Dict[str, Any]]:
    for message in reversed(st.session_state.messages):
        if message.get("role") == "assistant":
            return message
    return None


def expected_prompt_type(message: Dict[str, Any]) -> Optional[str]:
    text = (message.get("text") or "").lower()
    if "which destination are you planning to visit" in text:
        return "destination"
    if "where will you travel from" in text:
        return "origin"
    if "travel dates" in text or ("departure date" in text and "return date" in text):
        return "date_range"
    if "what is your departure date" in text:
        return "travel_date"
    if "what is your return date" in text:
        return "return_date"
    if "what is your approximate travel budget" in text or "what is your approximate budget" in text:
        return "budget"
    if "how strong is your sustainability preference" in text or "should sustainability preference be" in text:
        return "sustainability_level"
    return None


def build_prompt_response(message: Optional[Dict[str, Any]], user_text: str) -> Dict[str, str]:
    cleaned_text = user_text.strip()
    prompt_type = expected_prompt_type(message or {})

    if prompt_type == "destination":
        return {"raw": build_intent_payload("provide_destination", {"destination": cleaned_text}), "display": cleaned_text}
    if prompt_type == "origin":
        return {"raw": build_intent_payload("provide_origin", {"origin": cleaned_text}), "display": cleaned_text}
    if prompt_type == "travel_date":
        return {"raw": build_intent_payload("provide_dates", {"travel_date": cleaned_text}), "display": cleaned_text}
    if prompt_type == "return_date":
        return {"raw": build_intent_payload("provide_dates", {"return_date": cleaned_text}), "display": cleaned_text}
    if prompt_type == "budget":
        return {"raw": build_intent_payload("provide_budget", {"budget": cleaned_text}), "display": cleaned_text}
    if prompt_type == "sustainability_level":
        normalized_text = cleaned_text.lower()
        if normalized_text in {"low", "medium", "high"}:
            return {
                "raw": build_intent_payload(
                    "provide_sustainability_preference",
                    {"sustainability_level": normalized_text},
                ),
                "display": cleaned_text,
            }

    return {"raw": cleaned_text, "display": cleaned_text}


def queue_prompt_response(user_text: str) -> None:
    structured_message = build_prompt_response(get_latest_assistant_message(), user_text)
    queue_user_message(structured_message["raw"], display_message=structured_message["display"])


def format_date_range_display(start_date: date, end_date: date) -> str:
    return f"{start_date.strftime('%d %b %Y')} to {end_date.strftime('%d %b %Y')}"


def format_single_date_message(selected_date: date) -> str:
    return selected_date.strftime("%d %B %Y")


def format_single_date_display(selected_date: date) -> str:
    return selected_date.strftime("%d %b %Y")


def render_cards(cards: List[Dict[str, Any]]) -> None:
    for card in cards:
        with st.container(border=True):
            st.subheader(card.get("title", "Recommendation"))
            st.write(card.get("description", ""))
            col1, col2, col3 = st.columns(3)
            col1.metric("Estimated price", f"EUR {card.get('price_eur', 0)}")
            col2.metric("Carbon", f"{card.get('carbon_kg', 0)} kg CO2e")
            col3.metric("Score", card.get("score", 0))
            st.write(f"Carbon label: {label_badge(card.get('carbon_label'))}")


def render_date_picker(message_index: int, prompt_type: str) -> None:
    default_departure = date.today() + timedelta(days=14)
    default_return = default_departure + timedelta(days=3)

    if prompt_type == "date_range":
        with st.form(key=f"travel_dates_form_{message_index}", border=False):
            selected_dates = st.date_input(
                "Select departure and return dates",
                value=(default_departure, default_return),
                min_value=date.today(),
                key=f"travel_dates_{message_index}",
            )
            submitted = st.form_submit_button("Use selected dates")

        if submitted:
            if isinstance(selected_dates, (tuple, list)) and len(selected_dates) == 2:
                departure_date, return_date = selected_dates
                queue_user_message(
                    build_intent_payload(
                        "provide_dates",
                        {
                            "travel_date": format_single_date_message(departure_date),
                            "return_date": format_single_date_message(return_date),
                        },
                    ),
                    display_message=format_date_range_display(departure_date, return_date),
                )
                st.rerun()

            st.warning("Please select both a departure date and a return date.")
        return

    field_name = "departure" if prompt_type == "travel_date" else "return"
    button_label = f"Use selected {field_name} date"

    with st.form(key=f"{prompt_type}_form_{message_index}", border=False):
        selected_date = st.date_input(
            f"Select {field_name} date",
            value=default_departure if prompt_type == "travel_date" else default_return,
            min_value=date.today(),
            key=f"{prompt_type}_{message_index}",
        )
        submitted = st.form_submit_button(button_label)

    if submitted:
        queue_user_message(
            build_intent_payload("provide_dates", {prompt_type: format_single_date_message(selected_date)}),
            display_message=format_single_date_display(selected_date),
        )
        st.rerun()


def render_assistant_message(message: Dict[str, Any], message_index: int, is_active_turn: bool) -> None:
    with st.chat_message("assistant"):
        if message.get("text"):
            st.write(message["text"])

        custom = message.get("custom") or {}
        if custom.get("cards"):
            render_cards(custom["cards"])
        if custom.get("handover_required"):
            st.warning("Human handover requested. A context package has been prepared for the advisor.")
            st.json(custom.get("handover_context", {}))
        if custom.get("carbon_label"):
            st.info(f"Carbon label: {label_badge(custom['carbon_label'])}")

        prompt_type = expected_prompt_type(message)
        if is_active_turn and prompt_type in {"date_range", "travel_date", "return_date"}:
            render_date_picker(message_index, prompt_type)

        for button_index, button in enumerate(message.get("buttons") or []):
            payload = button.get("payload", button.get("title", ""))
            title = button.get("title", "Option")
            key = f"button_{message_index}_{button_index}_{payload}"
            if st.button(title, key=key):
                queue_user_message(payload, display_message=title)


if "sender_id" not in st.session_state:
    st.session_state.sender_id = f"streamlit_{uuid.uuid4().hex}"

if "messages" not in st.session_state:
    st.session_state.messages = []

if "pending_user_message" not in st.session_state:
    st.session_state.pending_user_message = None

if "bootstrapped" not in st.session_state:
    st.session_state.bootstrapped = False

if not st.session_state.bootstrapped:
    bootstrap_conversation()

with st.expander("Privacy notice", expanded=False):
    st.write(
        "This academic prototype uses only the trip information you type into the chat. "
        "Do not enter sensitive personal data. API keys belong in environment variables, not in the source code."
    )

with st.expander("Accessibility support", expanded=False):
    st.write("Carbon labels are shown as text. Buttons are optional because all actions can also be typed as plain text.")

if st.button("Start new conversation", key="reset_conversation"):
    reset_conversation()
    st.rerun()

for index, message in enumerate(st.session_state.messages):
    if message["role"] == "user":
        st.chat_message("user").write(message["text"])
    else:
        render_assistant_message(message, index, is_active_turn=index == len(st.session_state.messages) - 1)

prompt = st.chat_input("Ask about sustainable transport, eco hotels, carbon footprint, or local experiences")
if prompt:
    queue_prompt_response(prompt)

pending = st.session_state.pending_user_message
if pending:
    st.session_state.pending_user_message = None
    handle_user_message(pending["raw"], display_message=pending["display"])
    st.rerun()


Writing frontend/streamlit_app.py


In [19]:
%%writefile frontend/webchat_index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Eco Travel Advisor Webchat</title>
  <style>
    body {
      font-family: Arial, sans-serif;
      margin: 0;
      background: #f4faf7;
      color: #173b2f;
    }
    .page {
      max-width: 900px;
      margin: 40px auto;
      padding: 24px;
      background: #ffffff;
      border-radius: 16px;
      box-shadow: 0 8px 24px rgba(0, 0, 0, 0.08);
    }
    .badge {
      display: inline-block;
      padding: 6px 10px;
      border-radius: 999px;
      background: #dff5e8;
      font-weight: bold;
    }
  </style>
</head>
<body>
  <main class="page">
    <span class="badge">Eco Travel Advisor</span>
    <h1>Conversational Agent for Sustainable Tourism Planning</h1>
    <p>This page uses Rasa Webchat and connects to the local Rasa REST endpoint.</p>
    <p>Privacy: do not enter sensitive personal information in this academic demo.</p>
    <p>Accessibility: carbon labels should always be written as text as well as shown with colour.</p>
  </main>

  <script>
    !(function () {
      let e = document.createElement("script"), t = document.head || document.getElementsByTagName("head")[0];
      e.src = "https://cdn.jsdelivr.net/npm/rasa-webchat/lib/index.js";
      e.async = true;
      e.onload = () => {
        window.WebChat.default({
          title: "Eco Travel Advisor",
          subtitle: "Sustainable tourism planning",
          initPayload: "/greet",
          socketUrl: "http://localhost:5005",
          socketPath: "/socket.io/",
          customData: { language: "en" },
          params: { storage: "session" },
          showFullScreenButton: true,
        }, null);
      };
      t.insertBefore(e, t.firstChild);
    })();
  </script>
</body>
</html>


Writing frontend/webchat_index.html


In [20]:
%%writefile tests/conftest.py
from pathlib import Path
import sys


ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


Writing tests/conftest.py


In [21]:
%%writefile tests/test_actions.py
from actions.actions import (
    ActionCollectTripPreferences,
    ActionPackageHandoverContext,
    ActionRecommendHotels,
    ActionTwoStageClarification,
    ClimatiqClient,
    RecommendationEngine,
)


class DummyDispatcher:
    def __init__(self):
        self.messages = []

    def utter_message(self, **kwargs):
        self.messages.append(kwargs)


class DummyTracker:
    def __init__(self, slots=None, latest_text="latest message"):
        self.slots = slots or {}
        self.latest_message = {"text": latest_text}

    def get_slot(self, key):
        return self.slots.get(key)


def get_slot_value(events, name):
    for event in events:
        if event.get("event") == "slot" and event.get("name") == name:
            return event.get("value")
    return None


def has_followup_action(events, action_name):
    return any(event.get("event") == "followup" and event.get("name") == action_name for event in events)


def test_recommendation_weights_change_with_sustainability_level():
    options = [
        {"id": "cheap_high_carbon", "name": "Cheap Flight", "price": 30, "carbon_kg": 140, "relevance": 0.8},
        {"id": "green_train", "name": "Green Train", "price": 90, "carbon_kg": 20, "relevance": 0.9},
    ]
    high = RecommendationEngine.rank_options(options, "high", 200, "transport")
    assert high[0]["id"] == "green_train"


def test_climatiq_missing_key_uses_mock_fallback(monkeypatch):
    monkeypatch.delenv("CLIMATIQ_API_KEY", raising=False)
    client = ClimatiqClient(api_key=None)
    result = client.estimate_transport("train", origin="Berlin", destination="Amsterdam")
    assert result["source"] == "mock_emission_factor"
    assert result["carbon_kg"] > 0
    assert "missing" in result["fallback_reason"]


def test_handover_context_contains_required_fields():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(
        slots={
            "destination": "Amsterdam",
            "origin": "Berlin",
            "travel_date": "12 June",
            "return_date": "15 June",
            "budget": "700",
            "sustainability_level": "high",
            "transport_mode": "train",
            "recommended_options": {"transport": []},
        },
        latest_text="I need a human advisor",
    )
    events = ActionPackageHandoverContext().run(dispatcher, tracker, {})
    summary = get_slot_value(events, "conversation_summary")
    assert summary["destination"] == "Amsterdam"
    assert summary["origin"] == "Berlin"
    assert summary["latest_user_message"] == "I need a human advisor"
    assert get_slot_value(events, "handover_required") is True


def test_collect_trip_preferences_requests_next_missing_slot():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(slots={"destination": "Amsterdam"})
    events = ActionCollectTripPreferences().run(dispatcher, tracker, {})
    assert events == []
    assert dispatcher.messages[0]["text"] == "Where will you travel from?"


def test_collect_trip_preferences_confirms_when_complete():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(
        slots={
            "destination": "Amsterdam",
            "origin": "Berlin",
            "travel_date": "12 June",
            "return_date": "15 June",
            "budget": "700",
            "sustainability_level": "high",
        }
    )
    events = ActionCollectTripPreferences().run(dispatcher, tracker, {})
    assert events == []
    assert dispatcher.messages[0]["text"] == "Thank you. I have enough information to prepare sustainable recommendations."
    assert "Trip summary: Berlin to Amsterdam" in dispatcher.messages[1]["text"]


def test_two_stage_clarification_first_attempt_asks_options():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(slots={"clarification_attempts": 0})
    events = ActionTwoStageClarification().run(dispatcher, tracker, {})
    assert dispatcher.messages
    assert dispatcher.messages[0]["buttons"]
    assert get_slot_value(events, "clarification_attempts") == 1


def test_two_stage_clarification_second_attempt_triggers_handover():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(slots={"clarification_attempts": 1})
    events = ActionTwoStageClarification().run(dispatcher, tracker, {})
    assert has_followup_action(events, "action_package_handover_context")
    assert get_slot_value(events, "handover_required") is True


def test_recommend_hotels_returns_cards():
    dispatcher = DummyDispatcher()
    tracker = DummyTracker(
        slots={
            "destination": "Amsterdam",
            "budget": "700",
            "sustainability_level": "high",
            "accommodation_type": "eco hotel",
        }
    )
    events = ActionRecommendHotels().run(dispatcher, tracker, {})
    assert dispatcher.messages
    assert dispatcher.messages[0]["json_message"]["cards"]
    assert get_slot_value(events, "selected_option") is not None


Writing tests/test_actions.py


In [22]:
%%writefile README.md
# Eco Travel Advisor

## Project overview

Eco Travel Advisor is a Rasa Open Source chatbot for the MSc assignment titled **Eco Travel Advisor: Conversational Agent for Sustainable Tourism Planning using the Rasa Platform**.

The assistant helps users plan sustainable trips by recommending lower carbon transport, eco friendly accommodation, responsible cultural experiences, and carbon offset options. It includes custom Python actions, mock data fallbacks, external API integration structure, fallback clarification, human advisor handover, tests, Docker deployment, and a simple Streamlit frontend.

## Assignment alignment

| Requirement | Where implemented |
|---|---|
| Rasa Open Source | `config.yml`, `domain.yml`, `data/` |
| Rasa NLU and Rasa Core | NLU examples, stories, rules, slots, policies |
| Multi turn dialogue with slots | `domain.yml`, `data/stories.yml`, `ActionCollectTripPreferences` |
| Custom actions | `actions/actions.py` |
| Climatiq and Amadeus integration structure | `ClimatiqClient`, `AmadeusClient` |
| Mock fallback data | `actions/mock_data/` |
| Two stage fallback | `ActionTwoStageClarification` |
| Human advisor handover | `ActionPackageHandoverContext` |
| Frontend | `frontend/streamlit_app.py`, `frontend/webchat_index.html` |
| Testing | `data/tests/test_stories.yml`, `tests/test_actions.py` |
| Docker deployment | `docker/` |
| Documentation | `README.md`, `docs/` |

## System architecture

1. **Rasa NLU layer** identifies user intents and extracts entities such as destination, origin, dates, budget, transport mode, and sustainability level.
2. **Rasa Core dialogue layer** uses stories, rules, slots, and policies to manage multi turn conversations.
3. **Custom action server** performs recommendation ranking, carbon estimation, fallback logic, and human handover packaging.
4. **External API layer** provides placeholders for Climatiq and Amadeus, with robust fallback to local JSON data.
5. **Frontend layer** offers a Streamlit interface and a Rasa Webchat HTML alternative.

## Setup instructions

Recommended environment: Python 3.10.

```bash
python -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
```

On Windows PowerShell:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r requirements.txt
```

## Environment variables

Copy the example file:

```bash
cp .env.example .env
```

Then add API keys if available:

```bash
CLIMATIQ_API_KEY=your_key_here
AMADEUS_CLIENT_ID=your_client_id_here
AMADEUS_CLIENT_SECRET=your_client_secret_here
```

The chatbot works without these keys by using local mock data.

## Train the model

```bash
rasa data validate
rasa train
```

## Run the action server

Open a first terminal:

```bash
rasa run actions
```

The action server runs on `http://localhost:5055`.

## Run the Rasa server

Open a second terminal:

```bash
rasa run --enable-api --cors "*" --debug
```

The Rasa REST webhook runs on:

```text
http://localhost:5005/webhooks/rest/webhook
```

## Run the frontend

Open a third terminal:

```bash
streamlit run frontend/streamlit_app.py
```

Then open:

```text
http://localhost:8501
```

## Try one complete conversation

Example:

```text
User: Hello
Bot: Hello. I am your Eco Travel Advisor...
User: I want to plan a sustainable trip
Bot: Which destination are you planning to visit?
User: I want to visit Amsterdam
Bot: Where will you travel from?
User: I am travelling from Berlin
Bot: What are your travel dates?
User: from 12 June to 15 June
Bot: What is your approximate travel budget?
User: my budget is 700 euros
Bot: How strong is your sustainability preference: low, medium, or high?
User: high
User: show me low carbon transport options
Bot: Provides ranked transport recommendation cards.
```

## Testing commands

```bash
rasa data validate
rasa train
rasa test nlu
rasa test core --stories data/tests/test_stories.yml
pytest tests/test_actions.py
```

## Docker deployment

Build and start all services. The Rasa container validates training data, trains a model, then starts the server with Docker specific action endpoint settings:

```bash
docker compose -f docker/docker-compose.yml up --build
```

Services:

| Service | Port |
|---|---:|
| Rasa server | 5005 |
| Action server | 5055 |
| Streamlit frontend | 8501 |

For a faster repeat run, keep the generated `models/` folder mounted through Docker Compose.

## Limitations

This is an academic prototype. Climatiq and Amadeus integrations include defensive placeholder logic and mock fallback data. Real sustainability certifications, live pricing, availability, accessibility details, and carbon factors should be verified before real user deployment.

## Ethical considerations

The assistant should not present mock recommendations as real bookings. It should explain carbon estimates as approximate. It should avoid greenwashing by showing why each option is considered sustainable and where the data came from.

## GDPR and privacy notes

The prototype should collect only travel planning data needed for the conversation. API keys must be stored in environment variables. Do not commit `.env`. Do not ask users for passport numbers, payment details, medical details, or other sensitive information.

## Accessibility considerations

The frontend uses text labels in addition to colour coded carbon indicators. Students should test keyboard navigation, readable contrast, screen reader compatibility, and clear error messages.


Writing README.md


In [24]:
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def secret_or_blank(name: str) -> str:
    if userdata is None:
        return ''
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

climatiq_api_key = secret_or_blank('CLIMATIQ_API_KEY')
amadeus_client_id = secret_or_blank('AMADEUS_CLIENT_ID')
amadeus_client_secret = secret_or_blank('AMADEUS_CLIENT_SECRET')

env_lines = [
    f'CLIMATIQ_API_KEY={climatiq_api_key}',
    f'AMADEUS_CLIENT_ID={amadeus_client_id}',
    f'AMADEUS_CLIENT_SECRET={amadeus_client_secret}',
    'RASA_REST_URL=http://localhost:5005/webhooks/rest/webhook',
]
(PROJECT_ROOT / '.env').write_text(''.join(env_lines) + '', encoding='utf-8')

print('Created .env file in', PROJECT_ROOT)
print('CLIMATIQ_API_KEY set:', bool(climatiq_api_key))
print('AMADEUS_CLIENT_ID set:', bool(amadeus_client_id))
print('AMADEUS_CLIENT_SECRET set:', bool(amadeus_client_secret))


Created .env file in /content/eco_travel_advisor_project
CLIMATIQ_API_KEY set: False
AMADEUS_CLIENT_ID set: False
AMADEUS_CLIENT_SECRET set: False


In [25]:
from pathlib import Path

for path in sorted(PROJECT_ROOT.rglob('*')):
    if path.is_file() and '__pycache__' not in path.parts:
        print(path.relative_to(PROJECT_ROOT))


.env
.env.example
README.md
actions/__init__.py
actions/actions.py
actions/mock_data/carbon_offsets.json
actions/mock_data/cultural_experiences.json
actions/mock_data/hotels.json
actions/mock_data/transport_options.json
config.yml
credentials.yml
data/nlu.yml
data/rules.yml
data/stories.yml
data/tests/test_stories.yml
domain.yml
endpoints.yml
frontend/streamlit_app.py
frontend/webchat_index.html
requirements.txt
tests/conftest.py
tests/test_actions.py


## Install Dependencies


In [27]:
import os

ECO_PYTHON = os.environ['ECO_PYTHON']
ECO_PIP = os.environ['ECO_PIP']
BASE_ENV = os.environ.copy()
BASE_ENV['PYTHONUNBUFFERED'] = '1'
BASE_ENV['TMPDIR'] = os.environ['TMPDIR']

run_command([ECO_PIP, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], env=BASE_ENV)
run_command([ECO_PIP, 'install', '-r', 'requirements.txt'], env=BASE_ENV)


$ /content/miniforge3/envs/eco-rasa/bin/pip install --upgrade pip setuptools wheel


FileNotFoundError: [Errno 2] No such file or directory: '/content/miniforge3/envs/eco-rasa/bin/pip'

## Validate, Test, and Train


In [ ]:
run_command([os.environ['ECO_PYTHON'], '-m', 'rasa', 'data', 'validate'], env=BASE_ENV)


In [ ]:
run_command([os.environ['ECO_PYTHON'], '-m', 'pytest', '-q', 'tests/test_actions.py'], env=BASE_ENV)


In [ ]:
run_command([os.environ['ECO_PYTHON'], '-m', 'rasa', 'train'], env=BASE_ENV)


## Start the Services


In [ ]:
import time
import urllib.request
import subprocess

LOG_DIR = PROJECT_ROOT / 'logs'
SERVICE_HANDLES = {}


def stop_existing_services():
    handles = globals().get('SERVICE_HANDLES', {})
    for name, info in handles.items():
        process = info['process']
        if process.poll() is None:
            print(f'Stopping {name} (pid={process.pid})')
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
    globals()['SERVICE_HANDLES'] = {}


def start_service(name, command, stdout_name, stderr_name):
    stdout_path = LOG_DIR / stdout_name
    stderr_path = LOG_DIR / stderr_name
    stdout_handle = open(stdout_path, 'w', encoding='utf-8')
    stderr_handle = open(stderr_path, 'w', encoding='utf-8')
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        env=BASE_ENV,
        stdout=stdout_handle,
        stderr=stderr_handle,
    )
    SERVICE_HANDLES[name] = {
        'process': process,
        'stdout_path': stdout_path,
        'stderr_path': stderr_path,
        'stdout_handle': stdout_handle,
        'stderr_handle': stderr_handle,
    }
    print(f'Started {name} with pid={process.pid}')


def wait_for_http(url, timeout=240):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                body = response.read().decode('utf-8', errors='replace')
                print(f'{url} -> {response.status}')
                return response.status, body
        except Exception as exc:
            last_error = exc
            time.sleep(3)
    raise RuntimeError(f'Service at {url} did not become ready. Last error: {last_error}')


stop_existing_services()
start_service(
    'actions',
    [os.environ['ECO_PYTHON'], '-m', 'rasa_sdk', '--actions', 'actions'],
    'actions.out.log',
    'actions.err.log',
)
wait_for_http('http://127.0.0.1:5055/health')

start_service(
    'rasa',
    [
        os.environ['ECO_PYTHON'],
        '-m',
        'rasa',
        'run',
        '--enable-api',
        '--cors',
        '*',
        '--credentials',
        'credentials.yml',
        '--endpoints',
        'endpoints.yml',
    ],
    'rasa.out.log',
    'rasa.err.log',
)
wait_for_http('http://127.0.0.1:5005/status')

start_service(
    'streamlit',
    [
        os.environ['ECO_PYTHON'],
        '-m',
        'streamlit',
        'run',
        'frontend/streamlit_app.py',
        '--server.address',
        '0.0.0.0',
        '--server.port',
        '8501',
        '--server.headless',
        'true',
    ],
    'streamlit.out.log',
    'streamlit.err.log',
)
wait_for_http('http://127.0.0.1:8501')


In [ ]:
for name, info in SERVICE_HANDLES.items():
    print(f"{name}: pid={info['process'].pid}")
    print('  stdout ->', info['stdout_path'])
    print('  stderr ->', info['stderr_path'])


## Open the Streamlit Frontend


In [ ]:
from google.colab import output

try:
    output.serve_kernel_port_as_window(8501)
except AttributeError:
    output.serve_kernel_port_as_iframe(8501, height=1000)


## Smoke Test the REST Conversation


In [ ]:
import json
import uuid
import urllib.request

sender_id = f'colab_demo_{uuid.uuid4().hex[:8]}'
conversation = [
    '/greet',
    '/start_trip_planning',
    '/provide_destination{"destination":"Vienna"}',
    '/provide_origin{"origin":"Munich"}',
    '/provide_dates{"travel_date":"27 May 2026","return_date":"30 May 2026"}',
    '/provide_budget{"budget":"150"}',
    '/provide_sustainability_preference{"sustainability_level":"medium"}',
    '/provide_transport_mode{"transport_mode":"train"}',
    '/ask_carbon_footprint',
]

for message in conversation:
    payload = json.dumps({'sender': sender_id, 'message': message}).encode('utf-8')
    request = urllib.request.Request(
        'http://127.0.0.1:5005/webhooks/rest/webhook',
        data=payload,
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        replies = json.loads(response.read().decode('utf-8'))
    print('
USER:', message)
    for reply in replies:
        if 'text' in reply:
            print('BOT:', reply['text'])
        if 'custom' in reply:
            print('CUSTOM:', json.dumps(reply['custom'], indent=2)[:1200])


## Optional: Tail the Service Logs


In [ ]:
from pathlib import Path

for log_name in ['actions.err.log', 'rasa.err.log', 'streamlit.err.log']:
    log_path = PROJECT_ROOT / 'logs' / log_name
    print(f'===== {log_name} =====')
    if log_path.exists():
        print(log_path.read_text(encoding='utf-8', errors='replace')[-4000:])
    else:
        print('Log file not found.')


## Optional: Stop the Background Services


In [ ]:
stop_existing_services()
